# Clustering Analysis: Order Risk Segmentation

This notebook demonstrates a **K-Means clustering pipeline** applied to supply-chain orders.
The goal is to segment orders into risk cohorts — groups with distinct delivery-risk profiles —
enabling targeted interventions rather than one-size-fits-all policies.

## Why clustering?

- Orders arrive with heterogeneous attributes (shipping mode, scheduled days, region, discount level).
- Treating all orders identically ignores this structure.
- Clustering surfaces *natural groupings* that correlate with late-delivery risk.

## Hexagonal architecture note

The domain layer (`domain/`) stays pure: `Order` entities and `extract_features` live there
with **zero infrastructure dependencies**.
Clustering is an *adapter concern*: `KMeansClusterer` in `adapters/ml/` implements the
`ClustererPort` protocol.
Use-cases in `application/use_cases.py` orchestrate the pipeline without importing sklearn directly.

```
Domain           Application         Adapters
────────────     ─────────────────   ───────────────────────
Order ──────────→ FitClustersUseCase → KMeansClusterer (sklearn)
extract_features  ProfileClustersUseCase → FeatureEncoder
                                       DataCoCSVRepository
```


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from adapters.data.csv_repository import DataCoCSVRepository
from adapters.ml.feature_encoder import FeatureEncoder
from adapters.ml.kmeans_clusterer import KMeansClusterer
from application.use_cases import FitClustersUseCase, ProfileClustersUseCase


In [ ]:
repo = DataCoCSVRepository(PROJECT_ROOT / "data" / "sample" / "sample.csv")
orders = repo.get_orders()
print(f"Loaded {len(orders)} orders")


In [ ]:
encoder = FeatureEncoder()
fit_uc = FitClustersUseCase(data_repo=repo, feature_encoder=encoder)
result = fit_uc.execute(k_range=range(2, 11))
print(f"Optimal K by silhouette: {result.optimal_k}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ks = list(result.elbow_data.keys())
ax1.plot(ks, [result.elbow_data[k] for k in ks], 'bo-')
ax1.set_xlabel('K')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax2.plot(ks, [result.silhouette_scores[k] for k in ks], 'ro-')
ax2.set_xlabel('K')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "clustering_elbow.png", dpi=100)
plt.show()


## K Selection Rationale

Two complementary signals guide K selection:

| Signal | What it measures | Ideal value |
|--------|-----------------|-------------|
| **Inertia (elbow)** | Within-cluster sum-of-squares — compactness | Inflection point ("elbow") |
| **Silhouette score** | How well each point fits its cluster vs. neighbours | Closest to +1.0 |

We choose the **K with the highest silhouette score** because:

1. The elbow is often ambiguous — silhouette gives a single, objective number.
2. Silhouette penalises both over-splitting (too many tiny clusters) and under-splitting
   (heterogeneous clusters), making it well-suited to business segmentation.
3. For supply-chain risk, a well-separated cluster structure means each cohort gets a
   meaningfully different risk label — actionable for operations teams.

The elbow plot is still shown as a sanity check: a dramatic inertia drop before the chosen K
would suggest we under-clustered.


In [ ]:
encoder2 = FeatureEncoder()
clusterer = KMeansClusterer(n_clusters=result.optimal_k)
profile_uc = ProfileClustersUseCase(data_repo=repo, feature_encoder=encoder2, clusterer=clusterer)
cohorts = profile_uc.execute()


In [ ]:
cohort_df = pd.DataFrame([
    {
        "Cluster": c.cluster_id,
        "Label": c.label,
        "Size": c.size,
        "Late Rate": f"{c.late_rate:.1%}",
        "Dominant Mode": c.dominant_shipping_mode,
        "Avg Scheduled Days": c.avg_scheduled_days,
    }
    for c in cohorts
])
print(cohort_df.to_string(index=False))


In [ ]:
from domain.services import extract_features
from application.use_cases import CLUSTERING_NUMERIC_KEYS

raw_features = [extract_features(o) for o in orders]
clustering_features = [
    {k: v for k, v in f.items() if k in CLUSTERING_NUMERIC_KEYS or k == "shipping_mode"}
    for f in raw_features
]
encoder3 = FeatureEncoder()
X = encoder3.fit_transform(clustering_features)

clusterer2 = KMeansClusterer(n_clusters=result.optimal_k)
clusterer2.fit(X.values)
labels = clusterer2.get_labels()

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.values)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set2', alpha=0.7)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Order Clusters (PCA projection)')
plt.savefig(PROJECT_ROOT / "notebooks" / "clustering_pca.png", dpi=100)
plt.show()


## Business Interpretation

The clusters map onto distinct operational risk profiles. Typical patterns found in supply-chain
order data (exact labels depend on what the silhouette analysis selects):

| Cluster label | Typical characteristics | Risk implication |
|--------------|------------------------|------------------|
| **High-Risk Express** | Standard Class + high discount + long scheduled window | Late rate often >60%. Discount pressure may indicate expedited mis-routing. Prioritise carrier audits. |
| **Low-Risk Fast** | First Class / Second Class + short scheduled window + low discount | Late rate <20%. These orders self-select into reliable lanes. Maintain status quo. |
| **Bulk Slow** | Standard Class + high quantity + long scheduled window | Moderate-high late rate. Volume buffers cost but strains warehouse capacity during peaks. Consider safety-stock triggers. |
| **Premium Reliable** | Same Day / First Class + medium discount | Late rate lowest across cohorts. High-value orders that get carrier priority. Use as benchmark SLA. |

### Operational recommendations

1. **Targeted re-routing**: Assign High-Risk Express orders to First Class proactively when
   inventory allows — the unit cost increase is likely lower than the cost of a late penalty.
2. **Discount guardrails**: Deep discounts cluster with high-risk orders. A business rule
   capping discount on Standard Class shipments could shift orders toward safer cohorts.
3. **Lead-time buffers**: Bulk Slow orders benefit from +1 day scheduled buffer to absorb
   warehouse processing spikes.
4. **SLA benchmarking**: Use Premium Reliable cohort metrics as the gold-standard SLA target
   and communicate them in customer contracts.

> **Next step**: Feed cluster labels as a feature into the supervised late-delivery predictor
> (`LateDeliveryRiskPredictor`) to capture non-linear cohort effects beyond the individual
> feature values.
